In [1]:
!pip install -U crewai pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of instructor to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 4.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This cou

In [2]:
from google.colab import files
uploaded = files.upload()

Saving ind_as_infra_style_dummy_balance_sheet.csv to ind_as_infra_style_dummy_balance_sheet.csv


In [3]:
import pandas as pd

df = pd.read_csv("ind_as_infra_style_dummy_balance_sheet.csv")
df


,Category,Subcategory,Line_Item,Amount_INR_Crores
0,Assets,Non-Current,Property Plant and Equipment,1850
1,Assets,Non-Current,Capital Work in Progress,420
2,Assets,Non-Current,Right of Use Assets,210
3,Assets,Non-Current,Intangible Assets,95
4,Assets,Non-Current,Financial Assets - Investments,360
5,Assets,Non-Current,Deferred Tax Assets,140
6,Assets,Current,Inventories,520
7,Assets,Current,Trade Receivables,880
8,Assets,Current,Cash and Cash Equivalents,260
9,Assets,Current,Bank Balances other than Cash,190


In [4]:
import os
from google.colab import userdata
os.environ["OPENAI_MODEL_NAME"] = "gpt-4o-mini"
os.environ["OPENAI_API_KEY"]=userdata.get('OPENAI_API_KEY')

In [5]:
from crewai import Agent

planner = Agent(
    role="Financial Analysis Planner",
    goal="Plan a structured and comprehensive balance sheet analysis",
    backstory=(
        "You are a CFO who understands how to break down balance sheet "
        "analysis into logical analytical steps."
    ),
    verbose=True
)

validator = Agent(
    role="Data Validation and Structuring Agent",
    goal="Ensure balance sheet data is complete, balanced, and Ind AS compliant",
    backstory=(
        "You are a statutory auditor responsible for validating financial "
        "statements before analysis."
    ),
    verbose=True
)

liquidity_analyst = Agent(
    role="Liquidity and Working Capital Analyst",
    goal="Assess short-term liquidity and working capital health",
    backstory=(
        "You are a treasury expert focused on liquidity risk and cash management."
    ),
    verbose=True
)

solvency_analyst = Agent(
    role="Solvency and Capital Structure Analyst",
    goal="Evaluate long-term financial stability and leverage",
    backstory=(
        "You are a credit rating analyst evaluating debt sustainability."
    ),
    verbose=True
)

narrator = Agent(
    role="Financial Narrator",
    goal="Translate financial analysis into executive-level insights",
    backstory=(
        "You specialize in explaining financial statements to CXOs, boards, "
        "and MBA students."
    ),
    verbose=True
)


In [6]:
from crewai import Task

planning_task = Task(
    description=(
        "Review the provided balance sheet data and determine the appropriate "
        "analytical steps required for a comprehensive financial analysis. "
        "Outline what aspects should be examined, including validation, "
        "liquidity, and solvency."
    ),
    expected_output=(
        "A clear analysis plan describing the sequence of financial checks "
        "and evaluations to be performed."
    ),
    agent=planner
)

validation_task = Task(
    description=(
        "Validate the balance sheet data by checking whether total assets "
        "equal total equity and liabilities. Identify any missing, "
        "misclassified, or unusual Ind AS line items. "
        f"\n\nBalance Sheet Data:\n{df.to_string(index=False)}"
    ),
    expected_output=(
        "A validation report confirming whether the balance sheet is balanced, "
        "along with any structural or classification issues identified."
    ),
    context=[planning_task],
    agent=validator
)

liquidity_analysis_task = Task(
    description=(
        "Analyze the short-term financial health of the company by evaluating "
        "current assets and current liabilities. Compute key liquidity metrics "
        "such as Current Ratio and Net Working Capital, and interpret the results."
        f"\n\nBalance Sheet Data:\n{df.to_string(index=False)}"
    ),
    expected_output=(
        "A liquidity analysis explaining the company’s ability to meet "
        "short-term obligations, supported by key metrics and interpretation."
    ),
    context=[validation_task],
    agent=liquidity_analyst
)

solvency_analysis_task = Task(
    description=(
        "Evaluate the company’s long-term financial stability by analyzing "
        "its capital structure. Focus on debt levels, equity base, "
        "and long-term obligations such as borrowings and lease liabilities."
        f"\n\nBalance Sheet Data:\n{df.to_string(index=False)}"
    ),
    expected_output=(
        "A solvency assessment discussing leverage, debt sustainability, "
        "and long-term financial risk."
    ),
    context=[validation_task],
    agent=solvency_analyst
)

narration_task = Task(
    description=(
        "Using the validated balance sheet data and the analytical insights "
        "produced by previous tasks, create a comprehensive executive summary "
        "of the company’s financial position. Clearly highlight key strengths, "
        "major risks, and the overall financial assessment. "
        "Do not fabricate any information beyond the analysis provided."
    ),
    expected_output=(
        "A well-structured executive summary written in Markdown format, "
        "including sections such as Key Strengths, Key Risks, and Overall "
        "Financial Assessment, suitable for CXOs and board members."
    ),
    output_file="balance_sheet_executive_summary.md",
    context=[
        validation_task,
        liquidity_analysis_task,
        solvency_analysis_task
    ],
    agent=narrator
)


In [7]:
from crewai import Crew

crew = Crew(
    agents=[
        planner,
        validator,
        liquidity_analyst,
        solvency_analyst,
        narrator
    ],
    tasks=[
        planning_task,
        validation_task,
        liquidity_analysis_task,
        solvency_analysis_task,
        narration_task
    ],
    verbose=True
)

result = crew.kickoff()


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  28bf79db-5c50-4775-8b3c-eb9fc4b16eb7                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Review the provided balance sheet data and determine the appropriate analytical steps required for a     │
│  comprehensive financial analysis. Outline what aspects should be examined, including validation, liquidity,    │
│  and solvency.                                                                                                  │
│  ID: 2d81871d-b8be-4eb5-b62e-3771dbbc2396                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Financial Analysis Planner                                                                              │
│                                                                                                                 │
│  Task: Review the provided balance sheet data and determine the appropriate analytical steps required for a     │
│  comprehensive financial analysis. Outline what aspects should be examined, including validation, liquidity,    │
│  and solvency.                                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Financial Analysis Planner                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  To conduct a comprehensive balance sheet analysis, I will follow a structured and detailed approach. Below is  │
│  the analysis plan outlining the necessary steps and aspects to be examined:                                    │
│                                                                                                                 │
│  1. **Initial Data Validation:**                                                                                │
│     - Ensure accuracy of the balance sheet data: cross-reference with the general ledger and previous periods.  │
│     - Check for adherence to accounting standards (e.g., GAAP or IFRS).                                         │
│     - Verify that all numbers are reconciled and complete, paying attention to any anomalies or discrepancies.  │
│                                                                                                                 │
│  2. **Asset Analysis:**                                                                                         │
│     - **Current Assets:**                                                                                       │
│       - Examine the composition (cash, accounts receivable, inventory) and analyze turnover ratios (e.g.,       │
│  current ratio, quick ratio).                                                                                   │
│       - Assess collectability of accounts receivable and age of inventory.                                      │
│     - **Non-Current Assets:**                                                                                   │
│       - Review property, plant, and equipment valuations; check for impairments.                                │
│       - Analyze intangible assets for any amortization or impairment needs.                                     │
│                                                                                                                 │
│  3. **Liability Analysis:**                                                                                     │
│     - **Current Liabilities:**                                                                                  │
│       - Evaluate accounts payable, short-term debt, and other current obligations.                              │
│       - Calculate current ratio and quick ratio to assess liquidity.                                            │
│     - **Non-Current Liabilities:**                                                                              │
│       - Review long-term debt arrangements and assess payment schedules.                                        │
│       - Analyze contingent liabilities and any off-balance sheet arrangements.                                  │
│                                                                                                                 │
│  4. **Equity Analysis:**                                                                                        │
│     - Review changes in equity section (retained earnings, additional paid-in capital).                         │
│     - Analyze dividends declared and their impact on retained earnings.                                         │
│     - Calculate return on equity (ROE) to gauge profitability relative to shareholder equity.                   │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Review the provided balance sheet data and determine the appropriate analytical steps required for a           │
│  comprehensive financial analysis. Outline what aspects should be examined, including validation, liquidity,    │
│  and solvency.                                                                                                  │
│  Agent:                                                                                                         │
│  Financial Analysis Planner                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Validate the balance sheet data by checking whether total assets equal total equity and liabilities.     │
│  Identify any missing, misclassified, or unusual Ind AS line items.                                             │
│                                                                                                                 │
│  Balance Sheet Data:                                                                                            │
│     Category Subcategory                         Line_Item  Amount_INR_Crores                                   │
│       Assets Non-Current      Property Plant and Equipment               1850                                   │
│       Assets Non-Current          Capital Work in Progress                420                                   │
│       Assets Non-Current               Right of Use Assets                210                                   │
│       Assets Non-Current                 Intangible Assets                 95                                   │
│       Assets Non-Current    Financial Assets - Investments                360                                   │
│       Assets Non-Current               Deferred Tax Assets                140                                   │
│       Assets     Current                       Inventories                520                                   │
│       Assets     Current                 Trade Receivables                880                                   │
│       Assets     Current         Cash and Cash Equivalents                260                                   │
│       Assets     Current     Bank Balances other than Cash                190                                   │
│       Assets     Current              Other Current Assets                150                                   │
│       Equity      Equity              Equity Share Capital                900                                   │
│       Equity      Equity Other Equity (Reserves & Surplus)               1850                                   │
│  Liabilities Non-Current                        Borrowings               1350                                   │
│  Liabilities Non-Current                 Lease Liabilities                180                                   │
│  Liabilities Non-Current                        Provisions                210                                   │
│  Liabilities Non-Current          Deferred Tax Liabilities                260                                   │
│  Liabilities     Current                        Borrowings                420                                   │
│  Liabilities     Current                    Trade Payables                610                                   │
│  Liabilities     Current       Other Financial Liabilities                370                                   │
│  Liabilities     Current                        Provisions                235                                   │
│  Liabilities     Current         Other Current Liabilities                300                                   │
│  ID: 12f6123a-0b8b-4f94-9985-888a86f666fc                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (12 outputs left out)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Financial Narrator                                                                                      │
│                                                                                                                 │
│  Task: Using the validated balance sheet data and the analytical insights produced by previous tasks, create a  │
│  comprehensive executive summary of the company’s financial position. Clearly highlight key strengths, major    │
│  risks, and the overall financial assessment. Do not fabricate any information beyond the analysis provided.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Financial Narrator                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Executive Summary of the Company’s Financial Position                                                        │
│                                                                                                                 │
│  ## Key Strengths                                                                                               │
│  - **Strong Asset Base:** The company boasts total assets amounting to **5075 INR Crores**, primarily driven    │
│  by significant investments in non-current assets totaling **3075 INR Crores**. This foundational asset base    │
│  provides a solid platform for operational capabilities and growth.                                             │
│  - **Positive Liquidity Position:** The liquidity analysis reveals a **Current Ratio of approximately 1.34**    │
│  and a **Net Working Capital of 65 INR Crores**. These indicators suggest that the company is positioned to     │
│  meet short-term obligations, reflecting a reasonable level of operational efficiency in managing current       │
│  assets relative to current liabilities.                                                                        │
│  - **Moderate Debt Levels:** With a Debt-to-Equity Ratio of **0.71**, the company indicates a balanced          │
│  approach to leveraging debt compared to equity, suggesting a moderated financial risk profile. This ratio      │
│  signals that the company has a strong equity cushion relative to its debt commitments.                         │
│                                                                                                                 │
│  ## Key Risks                                                                                                   │
│  - **Balance Sheet Imbalance:** The reported values indicate a critical imbalance where **Total Assets (5075    │
│  INR Crores)** do not equal **Total Liabilities and Equity (6685 INR Crores)**. This discrepancy necessitates   │
│  urgent reconciliation efforts to ensure accurate and trustworthy financial reporting.                          │
│  - **Potential Misclassification of Assets and Liabilities:** There is a risk of misclassification between      │
│  non-current and current categories, which could obscure true liquidity status and impact financial             │
│  decision-making. It is essential to critically review classifications as per Ind AS standards.                 │
│  - **Low Net Working Capital:** While the company possesses a positive Net Working Capital, the relatively low  │
│  figure raises concerns about liquidity constraints and the ability to respond to unforeseen expenses or        │
│  opportunities without seeking additional financing.                                                            │
│                                                                                                                 │
│  ## Overall Financial Assessment                                                                                │
│  The company presents a robust profile with commendable strengths in its asset structure and moderately         │
│  leveraged position. However, the fundamental issue of the unbalanced balance sheet poses significant risks     │
│  that must be addressed to enhance trust and clarity among stakeholders. Immediate steps should focus on        │
│  reconciling discrepancies, ensuring appropriate classi

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Using the validated balance sheet data and the analytical insights produced by previous tasks, create a        │
│  comprehensive executive summary of the company’s financial position. Clearly highlight key strengths, major    │
│  risks, and the overall financial assessment. Do not fabricate any information beyond the analysis provided.    │
│  Agent:                                                                                                         │
│  Financial Narrator                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  28bf79db-5c50-4775-8b3c-eb9fc4b16eb7                                                                           │
│  Final Output: # Executive Summary of the Company’s Financial Position                                          │
│                                                                                                                 │
│  ## Key Strengths                                                                                               │
│  - **Strong Asset Base:** The company boasts total assets amounting to **5075 INR Crores**, primarily driven    │
│  by significant investments in non-current assets totaling **3075 INR Crores**. This foundational asset base    │
│  provides a solid platform for operational capabilities and growth.                                             │
│  - **Positive Liquidity Position:** The liquidity analysis reveals a **Current Ratio of approximately 1.34**    │
│  and a **Net Working Capital of 65 INR Crores**. These indicators suggest that the company is positioned to     │
│  meet short-term obligations, reflecting a reasonable level of operational efficiency in managing current       │
│  assets relative to current liabilities.                                                                        │
│  - **Moderate Debt Levels:** With a Debt-to-Equity Ratio of **0.71**, the company indicates a balanced          │
│  approach to leveraging debt compared to equity, suggesting a moderated financial risk profile. This ratio      │
│  signals that the company has a strong equity cushion relative to its debt commitments.                         │
│                                                                                                                 │
│  ## Key Risks                                                                                                   │
│  - **Balance Sheet Imbalance:** The reported values indicate a critical imbalance where **Total Assets (5075    │
│  INR Crores)** do not equal **Total Liabilities and Equity (6685 INR Crores)**. This discrepancy necessitates   │
│  urgent reconciliation efforts to ensure accurate and trustworthy financial reporting.                          │
│  - **Potential Misclassification of Assets and Liabilities:** There is a risk of misclassification between      │
│  non-current and current categories, which could obscure true liquidity status and impact financial             │
│  decision-making. It is essential to critically review classifications as per Ind AS standards.                 │
│  - **Low Net Working Capital:** While the company possesses a positive Net Working Capital, the relatively low  │
│  figure raises concerns about liquidity constraints and the ability to respond to unforeseen expenses or        │
│  opportunities without seeking additional financing.                                                            │
│                                                                                                                 │
│  ## Overall Financial Assessment                                                                                │
│  The company presents a robust profile with commendable strengths in its asset structure and moderately         │
│  leveraged position. However, the fundamental issue of